In [ ]:
!pip install pandas scikit-learn

In [43]:
import json
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:

# 1. Read metrics.json and keep only the low-correlated metrics

# Load the low correlated metric names from CSV
low_corr_df = pd.read_csv("low_correlated_metric_names.csv")
selected_metrics = set(low_corr_df["metric_name"].tolist())

with open("../data/metrics.json", "r") as f:
    metrics_data = json.load(f)

# Parse the JSON structure into rows containing repository and filtered metrics
parsed_rows = []
for package_id, package_info in metrics_data.get("packages", {}).items():
    repo = package_info.get("repository")
    metrics = package_info.get("metrics", {})

    # Filter metrics to only include those in the allowed list
    filtered_metrics = {k: v for k, v in metrics.items() if k in selected_metrics}

    # Add repository to the record to allow joining later
    filtered_metrics["repository"] = repo
    parsed_rows.append(filtered_metrics)

# Create the metrics DataFrame
df_metrics = pd.DataFrame(parsed_rows)

In [45]:
df_metrics

,total_commits,pony_factor,contributor_growth_rate,days_since_last_commit,casual_regular_contributors_rate,commits_over_periods_rate,coefficient_of_variation,file_types_binary,message_size_mean,message_size_median,found_file_license,found_file_adopters,repository
0,0,0,0.0,NaN,0.000000,0.000,NaN,0,0.000,0,1,0,https://github.com/moment/moment.git
1,0,0,0.0,NaN,0.000000,0.000,NaN,0,0.000,0,1,0,https://github.com/epoberezkin/fast-json-stabl...
2,25,1,1.0,66.0,1.000000,0.240,1.939897,0,24.320,28,1,0,https://github.com/holepunchto/b4a.git
3,10,1,-0.5,30.0,1.000000,0.100,3.108054,0,44.900,30,1,0,https://github.com/lydell/js-tokens.git
4,0,0,0.0,NaN,0.000000,0.000,NaN,0,0.000,0,0,0,https://github.com/sindresorhus/array-union.git
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,1,1,-1.0,188.0,0.000000,0.000,3.464102,0,65.000,65,1,0,https://github.com/inspect-js/is-async-functio...
66,0,0,0.0,NaN,0.000000,0.000,NaN,0,0.000,0,1,0,https://github.com/mysticatea/npm-run-all.git
67,0,0,0.0,NaN,0.000000,0.000,NaN,0,0.000,0,1,0,https://github.com/lupomontero/psl.git
68,4,1,0.0,117.0,1.000000,0.000,2.669270,0,27.250,44,0,0,https://github.com/sindresorhus/decamelize.git


In [ ]:
df_expert = pd.read_csv("../data/expert-classification.csv")

df_filtered = df_expert[df_expert["Result"].isin(["Healthy", "Unhealthy"])]


# we just need the field to match with the metrics and the final category
df_expert_filtered = df_filtered[["downloadLocation", "Result"]].copy()

In [47]:
# 3. Standardize URL fields and merge datasets

def clean_repo_url(url):
    """Standardizes URLs by removing trailing slashes and '.git' extensions.
    """

    url = url.strip()
    if url.endswith("/"):
        url = url[:-1]
    if url.endswith(".git"):
        url = url[:-4]
    return url.lower()


# Apply the standardization to the matching keys
df_metrics["clean_url"] = df_metrics["repository"].apply(clean_repo_url)
df_expert_filtered["clean_url"] = df_expert_filtered["downloadLocation"].apply(clean_repo_url)

# Merge the dataframes on the cleaned URL column
# Using an inner join to keep only matching records (change to 'left' or 'outer' if needed)
df_merged = pd.merge(df_metrics, df_expert_filtered, on="clean_url", how="inner")

# Drop the temporary matching key
df_merged = df_merged.drop(columns=["repository", "downloadLocation"])

# Display the merged DataFrame summary
df_merged.head()

,total_commits,pony_factor,contributor_growth_rate,days_since_last_commit,casual_regular_contributors_rate,commits_over_periods_rate,coefficient_of_variation,file_types_binary,message_size_mean,message_size_median,found_file_license,found_file_adopters,clean_url,Result
0,0,0,0.0,NaN,0.0,0.00,NaN,0,0.00,0,1,0,https://github.com/moment/moment,Unhealthy
1,0,0,0.0,NaN,0.0,0.00,NaN,0,0.00,0,1,0,https://github.com/epoberezkin/fast-json-stabl...,Unhealthy
2,25,1,1.0,66.0,1.0,0.24,1.939897,0,24.32,28,1,0,https://github.com/holepunchto/b4a,Healthy
3,10,1,-0.5,30.0,1.0,0.10,3.108054,0,44.90,30,1,0,https://github.com/lydell/js-tokens,Healthy
4,0,0,0.0,NaN,0.0,0.00,NaN,0,0.00,0,0,0,https://github.com/sindresorhus/array-union,Unhealthy


### FIXME: Issues at this point

1. we identified NaN values in days_since_last_commit and coefficient_of_variation at this point
2. StandardScaler will calculate a standard deviation of 0 for the fields that have a 0 in every single row, like file_types_binary or found_file_adopters. It is recommended to drop them

In [48]:
## issue: we identified NaN values in days_since_last_commit and coefficient_of_variation at this point

x = df_merged.drop(columns=["clean_url","Result"])
y = df_merged["Result"]

# handle missing values (NaN) ---
imputer = SimpleImputer(strategy='constant', fill_value=0)
x_imputed = imputer.fit_transform(x)

# Rescale features (Mean = 0, Std Dev = 1)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_imputed)

x_scaled_df = pd.DataFrame(x_scaled, columns=x.columns)

# now we rebuild the dataframe with the missing columns
df_tracking = df_merged[['clean_url', 'Result']].reset_index(drop=True)
final_df = pd.concat([df_tracking, x_scaled_df], axis=1)

In [49]:
output_file = 'relevant_metrics_project.csv'
final_df.to_csv(output_file, index=False)